# 실습 2. 강화학습으로 균형 잡기 — MuJoCo + PPO + Sim-to-Real

**피지컬 AI 실습 시리즈 2/3** · CPU로 충분합니다 (학습 셀은 몇 분 걸릴 수 있음).

물리 엔진 **MuJoCo** 위에서 카트 위 막대를 세우는 **InvertedPendulum** 과제를 풉니다.

| 단계 | 내용 | 핵심 질문 |
|---|---|---|
| 2-1 | 환경 살펴보기 | 관측(observation), 행동(action), 보상(reward)은 무엇인가? |
| 2-2 | 무작위 정책 vs 손으로 만든 제어기 | 사람이 설계한 규칙은 얼마나 잘하나? |
| 2-3 | PPO로 학습 | 보상만 주면 로봇이 스스로 배울 수 있나? |
| 2-4 | Sim-to-Real 갭 체험 | 실제 로봇처럼 반응이 40ms 늦으면? |
| 2-5 | 도메인 랜덤화 | 여러 지연을 섞어 학습하면 강건해지나? |

In [ ]:
# Colab에서는 아래 줄의 주석을 풀고 한 번 실행하세요.
# !pip install -q "gymnasium[mujoco]" stable-baselines3

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from matplotlib import animation
from IPython.display import HTML, display

ENV_ID = "InvertedPendulum-v5"
SEED = 0


# ---- 영상 도구: 카트와 막대를 그림으로 그려 움직임을 보여 줍니다 ----
def record_episode(policy, env, seed=1, max_steps=200, after_fall=35):
    """정책으로 한 판을 돌려 매 스텝 관측을 모읍니다. 쓰러진 뒤에는 모터를 끄고 막대가 넘어가는 모습을 조금 더 기록합니다."""
    obs, _ = env.reset(seed=seed)
    frames, fell_at = [obs], None
    for t in range(max_steps):
        action = policy(obs) if fell_at is None else np.zeros(env.action_space.shape)
        obs, _, terminated, _, _ = env.step(action)
        frames.append(obs)
        if terminated and fell_at is None:
            fell_at = t + 1
        if fell_at is not None and t + 1 - fell_at >= after_fall:
            break
    return np.array(frames), fell_at


def show_cartpoles(clips, titles, step_s=0.04):
    """clips: record_episode 결과 목록. 나란히 놓고 실제 속도로 재생합니다."""
    fig, axes = plt.subplots(1, len(clips), figsize=(3.6 * len(clips), 3.4))
    axes = np.atleast_1d(axes)
    parts = []
    for ax, (frames, fell_at), title in zip(axes, clips, titles):
        ax.plot([-1.1, 1.1], [0, 0], color="gray", lw=3)                  # 레일
        cart = plt.Rectangle((0, -0.06), 0.3, 0.12, color="tab:blue")
        ax.add_patch(cart)
        pole, = ax.plot([], [], lw=6, color="tab:orange", solid_capstyle="round")
        status = ax.text(0, -0.35, "", ha="center", fontsize=11, fontweight="bold")
        ax.set_title(title, fontsize=12)
        ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.45, 0.75); ax.set_aspect("equal"); ax.axis("off")
        parts.append((cart, pole, status, frames, fell_at))
    n = max(len(c[0]) for c in clips)

    def update(i):
        for cart, pole, status, frames, fell_at in parts:
            k = min(i, len(frames) - 1)
            x, th = frames[k][0], frames[k][1]
            cart.set_x(x - 0.15)
            pole.set_data([x, x + 0.6 * np.sin(th)], [0.06, 0.06 + 0.6 * np.cos(th)])
            if fell_at is not None and k >= fell_at:
                status.set_text(f"FELL at {fell_at * step_s:.2f} s"); status.set_color("red")
            else:
                status.set_text(f"balancing  {k * step_s:.1f} s"); status.set_color("green")

    ani = animation.FuncAnimation(fig, update, frames=n, interval=step_s * 1000)
    plt.close(fig)
    try:
        html = ani.to_html5_video()   # Colab에는 ffmpeg가 있어 가벼운 mp4로 만듦
    except Exception:
        html = ani.to_jshtml()        # ffmpeg가 없으면 프레임 묶음으로 대신 보여 줌
    display(HTML(html))

## 2-1. 환경 살펴보기

강화학습의 기본 틀은 간단합니다: 에이전트(로봇의 두뇌)가 **상태**를 보고 **행동**을 하면, 환경이 **다음 상태**와 **보상(점수)** 을 돌려줍니다.

- **관측** (4차원): 카트 위치, 막대 각도, 카트 속도, 막대 각속도
- **행동** (1차원): 카트에 가하는 힘, 범위 [-3, 3]
- **보상**: 막대가 쓰러지지 않은 매 스텝마다 +1 (최대 1000스텝 → 최대 점수 1000)

In [ ]:
env = gym.make(ENV_ID)
obs, info = env.reset(seed=SEED)
print("observation space:", env.observation_space)
print("action space     :", env.action_space)
print("first observation:", np.round(obs, 4))

print("control step:", env.unwrapped.dt, "s  (MuJoCo timestep x frame_skip)")

## 2-2. 기준선 만들기: 무작위 정책 vs 손으로 설계한 선형 제어기

정책(policy)은 "관측 → 행동" 함수입니다. 먼저 두 가지 정책을 비교합니다.
- **무작위 정책**: 아무 힘이나 줌
- **선형 제어기**: 관측값 4개에 각각 정해 둔 숫자(게인)를 곱해 더한 만큼 미는 규칙 — 실습 1의 PD 제어와 같은 발상 (막대가 기울어진 쪽으로 카트를 밀기)

In [ ]:
def run_episode(env, policy, seed=None, max_steps=1000):
    obs, _ = env.reset(seed=seed)
    total = 0.0
    for _ in range(max_steps):
        action = policy(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        total += reward
        if terminated or truncated:
            break
    return total


def evaluate(env, policy, n=10, seed0=100):
    returns = [run_episode(env, policy, seed=seed0 + i) for i in range(n)]
    return float(np.mean(returns)), float(np.std(returns))


random_policy = lambda obs: env.action_space.sample()

K = np.array([1.0, 10.0, 1.5, 1.5])  # [카트 위치, 막대 각도, 카트 속도, 막대 각속도]에 곱할 게인
linear_policy = lambda obs: np.clip([K @ obs], -3, 3)

print("random policy : mean return = %.1f ± %.1f" % evaluate(env, random_policy))
print("linear policy : mean return = %.1f ± %.1f" % evaluate(env, linear_policy))

### 🎬 두 정책 비교 영상
파란 상자가 카트, 주황 막대가 세워야 할 막대입니다. 처음 8초만 실제 속도로 보여 줍니다.
막대가 일정 각도 이상 기울면 그 판은 끝(FELL)입니다. 그 뒤에는 모터를 끄고 막대가 넘어가는 모습을 조금 더 보여 줍니다.

In [ ]:
show_cartpoles([record_episode(random_policy, gym.make(ENV_ID)),
                record_episode(linear_policy, gym.make(ENV_ID))],
               ["random", "hand-tuned linear"])

> **질문:** 위 게인 `K`는 사람이 몇 번 시행착오로 찾은 값입니다. 각 성분의 부호를 하나씩 뒤집으면 어떻게 되나요? 이런 튜닝을 관절이 30개인 휴머노이드에서 한다면?

## 2-3. PPO로 학습하기

**PPO (Proximal Policy Optimization)** 는 로봇 강화학습에서 가장 널리 쓰이는 알고리즘 중 하나입니다.
신경망 정책을 두고, **보상을 많이 받은 행동의 확률은 올리고** 나머지는 내리되, 한 번에 너무 크게 바뀌지 않도록(clip) 업데이트합니다.

우리는 규칙을 하나도 주지 않습니다. **"쓰러지지 않으면 +1"이라는 보상만** 줍니다.

In [ ]:
class Snapshots(BaseCallback):
    """학습 도중 정해 둔 시점마다 현재 정책으로 한 판을 녹화합니다 (학습 자체에는 영향 없음)."""

    def __init__(self, at_steps):
        super().__init__()
        self.at_steps, self.clips = list(at_steps), {}

    def _record(self, step):
        self.clips[step] = record_episode(lambda o: self.model.predict(o, deterministic=True)[0], gym.make(ENV_ID))

    def _on_training_start(self):
        self._record(0)

    def _on_step(self):
        while self.at_steps and self.num_timesteps >= self.at_steps[0]:
            self._record(self.at_steps.pop(0))
        return True


def train_ppo(make_env_fn, total_timesteps=60_000, seed=SEED, callback=None):
    train_env = Monitor(make_env_fn())
    agent = PPO("MlpPolicy", train_env, n_steps=1024, batch_size=64, learning_rate=3e-4,
                gamma=0.99, verbose=0, seed=seed)
    t0 = time.time()
    agent.learn(total_timesteps=total_timesteps, callback=callback)
    print(f"trained {total_timesteps} steps in {time.time() - t0:.0f}s")
    return agent, np.array(train_env.get_episode_rewards()), np.array(train_env.get_episode_lengths())


snapshots = Snapshots(at_steps=[4_000, 10_000])
agent, ep_rewards, ep_lengths = train_ppo(lambda: gym.make(ENV_ID), callback=snapshots)

steps = np.cumsum(ep_lengths)
smooth = np.convolve(ep_rewards, np.ones(10) / 10, mode="valid")
plt.figure(figsize=(8, 4))
plt.plot(steps, ep_rewards, alpha=0.3, label="episode return")
plt.plot(steps[9:], smooth, label="moving avg (10)")
plt.axhline(1000, color="k", ls="--", lw=1, label="max")
plt.xlabel("environment steps"); plt.ylabel("return"); plt.grid(alpha=0.3); plt.legend()
plt.title("PPO learning curve on InvertedPendulum")
plt.show()

ppo_policy = lambda obs: agent.predict(obs, deterministic=True)[0]
print("PPO policy    : mean return = %.1f ± %.1f" % evaluate(env, ppo_policy))

### 🎬 배우는 과정 영상
학습 전, 4천 스텝, 1만 스텝, 학습 끝(6만 스텝) 시점의 정책으로 같은 조건에서 한 판씩 녹화했습니다. 오른쪽으로 갈수록 오래 버티는지 보세요.
규칙은 하나도 알려 주지 않았고, "쓰러지지 않으면 +1" 점수만 주었습니다.

In [ ]:
steps_shown = list(snapshots.clips)
show_cartpoles([snapshots.clips[s] for s in steps_shown] + [record_episode(ppo_policy, gym.make(ENV_ID))],
               [f"after {s:,} steps" if s else "before training" for s in steps_shown] + ["after 60,000 steps"])

### (선택) 학습된 정책 영상 보기
Colab에서는 렌더링 전에 `MUJOCO_GL=egl` 설정이 필요합니다. 실패해도 실습 진행에는 지장이 없습니다.

In [ ]:
SHOW_VIDEO = False  # True로 바꾸면 영상을 만듭니다.
if SHOW_VIDEO:
    import os
    os.environ.setdefault("MUJOCO_GL", "egl")
    from matplotlib import animation
    from IPython.display import HTML, display

    venv = gym.make(ENV_ID, render_mode="rgb_array", width=320, height=240)
    obs, _ = venv.reset(seed=1)
    frames = []
    for _ in range(200):
        obs, r, term, trunc, _ = venv.step(ppo_policy(obs))
        frames.append(venv.render())
        if term or trunc:
            break
    fig = plt.figure(figsize=(4, 3)); plt.axis("off")
    im = plt.imshow(frames[0])
    ani = animation.FuncAnimation(fig, lambda i: im.set_data(frames[i]), frames=len(frames), interval=40)
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

## 2-4. Sim-to-Real 갭 체험 — "실제 로봇은 늦게 반응한다"

시뮬레이터에서는 정책이 행동을 내는 즉시 모터가 움직입니다. 하지만 실제 로봇에서는
카메라·센서 읽기 → 신경망 추론(예: Jetson 같은 엣지 보드) → 통신 → 모터 드라이버까지 **지연(latency)** 이 생깁니다.

이 환경의 한 스텝은 0.04초입니다. 행동이 **1스텝(40ms)**, **2스텝(80ms)** 늦게 적용되는 "현실"을 만들어 평가해 봅니다.

In [ ]:
import collections

STEP_MS = int(env.unwrapped.dt * 1000)   # 한 스텝 = 40ms


class DelayWrapper(gym.Wrapper):
    """행동을 delay 스텝 늦게 적용하는 래퍼. randomize=True면 에피소드마다 지연을 0~max_delay 중에서 무작위로 뽑음."""

    def __init__(self, env, delay=0, randomize=False, max_delay=2):
        super().__init__(env)
        self.delay, self.randomize, self.max_delay = delay, randomize, max_delay

    def reset(self, **kwargs):
        if self.randomize:
            self.delay = int(self.np_random.integers(0, self.max_delay + 1))
        self.buffer = collections.deque([np.zeros(self.action_space.shape, dtype=np.float32)] * self.delay)
        return self.env.reset(**kwargs)

    def step(self, action):
        self.buffer.append(np.asarray(action, dtype=np.float32))
        return self.env.step(self.buffer.popleft())   # delay 스텝 전의 행동이 지금 적용됨


DELAYS = [0, 1, 2]


def robustness(policy_fn, delays=DELAYS, n=5):
    return [evaluate(DelayWrapper(gym.make(ENV_ID), delay=d), policy_fn, n=n)[0] for d in delays]


rob_linear = robustness(linear_policy)
rob_ppo = robustness(ppo_policy)
for d, a, b in zip(DELAYS, rob_linear, rob_ppo):
    print(f"delay {d * STEP_MS:3d} ms: linear={a:7.1f}   PPO={b:7.1f}")

### 🎬 같은 PPO 정책, 지연만 다르게
정책은 완전히 같습니다. 바뀐 것은 **행동이 모터에 닿기까지 걸리는 시간**뿐입니다.

In [ ]:
show_cartpoles([record_episode(ppo_policy, DelayWrapper(gym.make(ENV_ID), delay=d)) for d in DELAYS],
               [f"PPO, delay {d * STEP_MS} ms" for d in DELAYS])

> **관찰:** 시뮬레이터에서 만점이던 PPO가 40ms 지연만으로 몇 스텝 만에 쓰러집니다. 사람이 만든 선형 제어기는 오히려 버팁니다.
> PPO는 "행동이 즉시 반영되는 세계"에 **과하게 최적화**되어, 세게 밀었다가 바로 반대로 미는 공격적인 전략을 배웠기 때문입니다.

## 2-5. 도메인 랜덤화 (Domain Randomization)

해결책 중 하나: **학습할 때부터 매번 다른 세계를 보여 주기.** 에피소드마다 지연을 0, 40, 80ms 중에서 무작위로 바꿔 학습합니다.
정책은 "특정 세계에 맞춘 요령" 대신 **여러 세계에서 통하는 전략**을 배우게 됩니다.

In [ ]:
agent_dr, _, _ = train_ppo(lambda: DelayWrapper(gym.make(ENV_ID), randomize=True, max_delay=2), total_timesteps=100_000)
dr_policy = lambda obs: agent_dr.predict(obs, deterministic=True)[0]
rob_dr = robustness(dr_policy)

x = np.arange(len(DELAYS)); w = 0.27
plt.figure(figsize=(8, 4))
plt.bar(x - w, rob_linear, w, label="hand-tuned linear")
plt.bar(x, rob_ppo, w, label="PPO (no delay in training)")
plt.bar(x + w, rob_dr, w, label="PPO + domain randomization")
plt.xticks(x, [f"{d * STEP_MS} ms" for d in DELAYS]); plt.xlabel("action delay"); plt.ylabel("mean return")
plt.axhline(1000, color="k", ls="--", lw=1); plt.grid(alpha=0.3, axis="y"); plt.legend()
plt.title("Robustness to control latency")
plt.show()

for d, a, b, c in zip(DELAYS, rob_linear, rob_ppo, rob_dr):
    print(f"delay {d * STEP_MS:3d} ms: linear={a:7.1f}   PPO={b:7.1f}   PPO+DR={c:7.1f}")

### 🎬 지연 40ms에서 다시 비교
왼쪽은 지연 없이 학습한 PPO, 가운데는 여러 지연을 섞어 학습한 PPO입니다. 오른쪽은 랜덤화 정책도 80ms에서는 버티지 못하는 모습입니다.

In [ ]:
show_cartpoles([record_episode(ppo_policy, DelayWrapper(gym.make(ENV_ID), delay=1)),
                record_episode(dr_policy, DelayWrapper(gym.make(ENV_ID), delay=1)),
                record_episode(dr_policy, DelayWrapper(gym.make(ENV_ID), delay=2))],
               ["PPO, delay 40 ms", "PPO + randomization, 40 ms", "PPO + randomization, 80 ms"])

**결과 해석**
- 40ms: 랜덤화로 학습한 정책은 회복합니다 → 학습 때 "현실의 불확실성"을 보여 주면 강건해집니다.
- 80ms: 셋 다 실패합니다. 이 모터 힘과 제어 주기로는 너무 늦은 것입니다. 랜덤화는 만능이 아니며, 이럴 땐
  **하드웨어(더 빠른 추론·통신)** 를 개선하거나, **최근 행동 이력을 관측에 넣어** 정책이 지연을 예측하게 해야 합니다.

### ✍️ 직접 해 보기 (도전 과제)
1. **지연 극복:** 관측에 "직전 2개의 행동"을 붙이는 래퍼를 만들어 80ms 지연에서 다시 학습해 보세요. 버틸 수 있나요?
2. **다른 현실 차이:** 모터 힘(`env.unwrapped.model.actuator_gear[0, 0]`)을 0.2배로 줄이거나, 관측에 `np.random.normal(0, 0.05, 4)` 노이즈를 더해 보세요.
3. **보상 설계:** 래퍼로 보상을 바꿔 "카트가 원점 근처에 머물수록 보너스"를 주면 행동이 어떻게 달라지나요?
4. **다른 로봇:** `ENV_ID`를 `"Reacher-v5"`(2관절 팔)나 `"HalfCheetah-v5"`로 바꿔 보세요. 학습 스텝이 훨씬 많이 필요합니다.

## 정리
- 강화학습은 **규칙 대신 보상**을 설계합니다. 설계자는 "무엇을"만 정하고 "어떻게"는 학습에 맡깁니다.
- 시뮬레이터는 안전하고 빠르고 싸지만 **현실과 다릅니다** (Sim-to-Real 갭).
- **도메인 랜덤화**는 이 갭을 줄이는 대표 기법이며, 실제 사족보행·휴머노이드 보행 학습에도 쓰입니다. 단, 만능은 아닙니다.
- 하지만 보상 설계가 어려운 작업(빨래 개기, 요리)은? → 실습 3: **사람의 시연에서 배우기**.